In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point

# ==========================================
# 1. LOAD DATASETS
# ==========================================
elem_df = pd.read_csv('Chicago_Public_Schools_–_Elementary_School_Progress_Report_Card_(2012-2013)_20260303.csv')
hs_df = pd.read_csv('Chicago_Public_Schools_–_High_School_Progress_Report_Card_(2012-2013)_20260303.csv')

community_gdf = gpd.read_file('chicago_community_boundaries.json')
community_gdf['area_numbe'] = community_gdf['area_numbe'].astype(int)

# ==========================================
# 2. CLEAN & FIX METRICS
# ==========================================
def clean_numeric(series):
    return pd.to_numeric(series.astype(str).str.replace('%', '', regex=False)
                                           .str.replace(',', '', regex=False)
                                           .replace('NDA', np.nan), errors='coerce')

# Elementary metrics
elem_df['Reading_Perf'] = clean_numeric(elem_df['Reading Performance All Grades'])
elem_df['Academic_Rank'] = elem_df['Reading_Perf'].rank(pct=True) * 100

# High School metrics
hs_df['ACT_Score'] = clean_numeric(hs_df['Average Score ACT 2012'])
hs_df['Academic_Rank'] = hs_df['ACT_Score'].rank(pct=True) * 100

# Common metrics
for df in [elem_df, hs_df]:
    df['Student_Attendance'] = clean_numeric(df['Student Attendance 2012 - Percent'])
    df['Teacher_Attendance'] = clean_numeric(df['Teacher Attendance 2012 - Percent'])
    df['Misconduct_Rate'] = clean_numeric(df['Misconducts Resulting in Suspensions 2012 - Percent'])
    df['Days_Suspended'] = clean_numeric(df['Average Days of Suspension 2012']) # Added as 6th EDA Variable
    
    # FIXING THE CPS DATASET ERROR: The dataset swapped Lat and Lon headers!
    df['True_Lon'] = pd.to_numeric(df['Latitude'], errors='coerce')
    df['True_Lat'] = pd.to_numeric(df['Longitude'], errors='coerce')

common_cols = ['School ID', 'School Name', 'Overall Rating', 'Student_Attendance', 
               'Teacher_Attendance', 'Misconduct_Rate', 'Days_Suspended', 'Academic_Rank', 'True_Lat', 'True_Lon']

comb_df = pd.concat([elem_df[common_cols], hs_df[common_cols]], ignore_index=True)
comb_df = comb_df.dropna(subset=['True_Lat', 'True_Lon'])

# Map Overall Rating to numerical values
rating_map = {'Level 1': 3, 'Level 2': 2, 'Level 3': 1, 'Not Enough Data': np.nan}
comb_df['Rating_Numeric'] = comb_df['Overall Rating'].map(rating_map)


# ==========================================
# 3. SPATIAL JOIN: MAP SCHOOLS TO COMMUNITIES
# ==========================================
geometry = [Point(xy) for xy in zip(comb_df['True_Lon'], comb_df['True_Lat'])]
schools_gdf = gpd.GeoDataFrame(comb_df, geometry=geometry, crs="EPSG:4326")

mapped_schools = gpd.sjoin(schools_gdf, community_gdf, how="inner", predicate="intersects")

# Label the specified areas
dangerous_areas = [25, 43, 23, 8, 29]
safest_areas = [74, 18, 12, 47, 9]

def label_safety(area):
    if area in dangerous_areas:
        return 'Top 5 Most Dangerous'
    elif area in safest_areas:
        return 'Top 5 Safest'
    else:
        return 'Other Areas'

mapped_schools['Neighborhood_Safety'] = mapped_schools['area_numbe'].apply(label_safety)
focus_df = mapped_schools[mapped_schools['Neighborhood_Safety'] != 'Other Areas'].copy()

# ==========================================
# 4. VISUALIZATIONS
# ==========================================
sns.set_theme(style="whitegrid")

# --- Visual 1: Underperformance by Race ---
race_cols = ['Reading Growth African American', 'Reading Growth Hispanic', 'Reading Growth White']
for col in race_cols:
    elem_df[col] = clean_numeric(elem_df[col])

race_df = elem_df.melt(value_vars=race_cols, var_name='Race', value_name='Reading_Growth').dropna()
race_df['Race'] = race_df['Race'].str.replace('Reading Growth ', '')

plt.figure(figsize=(10, 6))
sns.boxplot(x='Race', y='Reading_Growth', data=race_df, order=['African American', 'Hispanic', 'White'], palette='Set2')
plt.title('Reading Growth by Demographic (Performance Gap Analysis)', fontsize=14)
plt.ylabel('Reading Growth (%)')
plt.xlabel('Student Demographic')
plt.savefig('performance_by_race.png', bbox_inches='tight')
plt.close()

# --- Visual 2: Education Quality in Dangerous vs Safest Communities ---
plt.figure(figsize=(12, 6))
sns.barplot(x='area_numbe', y='Academic_Rank', hue='Neighborhood_Safety', 
            data=focus_df, dodge=False, palette=['#e74c3c', '#2ecc71'], errorbar=None)
plt.title('Academic Attainment Percentile in Specified Community Areas', fontsize=14)
plt.ylabel('Average Academic Rank Percentile')
plt.xlabel('Chicago Community Area Number')

# Legend moved outside the graph
plt.legend(title='Community Classification', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.savefig('safety_vs_education.png')
plt.close()

# --- Visual 3: EDA Correlation with 6 MEANINGFUL Variables ---
eda_vars = ['Student_Attendance', 'Teacher_Attendance', 'Misconduct_Rate', 
            'Days_Suspended', 'Academic_Rank', 'Rating_Numeric']
eda_df = comb_df[eda_vars].copy()

plt.figure(figsize=(9, 7))
sns.heatmap(eda_df.corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f")
plt.title('EDA Correlation: Chicago Education & Behavioral Factors', fontsize=14)
plt.tight_layout()
plt.savefig('eda_correlation_heatmap.png')
plt.close()

1. Neighborhood Safety is a Huge Factor
There is a massive academic gap between the 5 safest community areas (9, 12, 18, 47, 74) and the 5 most dangerous ones (8, 23, 25, 29, 43). Schools in safe neighborhoods rank way higher academically. It’s clear that outside neighborhood violence is bleeding into the classroom and hurting student success.

2. The Racial Performance Gap
When looking at reading growth, there’s a clear divide. Predominantly White student populations are seeing the most consistent reading growth, while Black and Hispanic students are showing lower median growth and a wider spread of underperformance.

3. What Actually Drives High Scores is Attendance: Student attendance is the first predictor of a high academic rank and overall school rating. If they show up, they score higher.